# Fine-tune Pegasus on SAMSum (Colab T4 GPU)

This notebook runs the **full** training (not smoke mode) on a free Colab GPU, then saves the model to Google Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

| Step | What happens | Rough time |
|---|---|---|
| 1 | Check the GPU | seconds |
| 2 | Mount Google Drive (asks you to approve: stay for this) | 30 s |
| 3 | Clone the repo, install packages | 2–4 min |
| 4 | Optional low-memory settings (off by default) | seconds |
| 5 | Ingestion + tokenization | 1–2 min |
| 6 | Training + validation loss (downloads Pegasus, ~2.3 GB) | 60–90 min |
| 7 | Save the model to Drive (or a zip to download) | 1–3 min |
| 8 | Summarize 2 test dialogues | 1 min |
| 9 | ROUGE evaluation + comparison with the base model | 5–10 min |

Keep this tab open while it trains. Free Colab disconnects idle sessions.

Every pipeline command runs as `!python ...` in a separate process, followed by an `assert` on its exit code.
That way a failure **stops Run all** instead of silently continuing, and each process frees its GPU memory when it exits.

In [ ]:
REPO_URL = "https://github.com/s4nkalp47/text-summarizer.git"
DRIVE_DIR = "/content/drive/MyDrive/text-summarizer"

## 1. Check the GPU

`nvidia-smi` lists the GPU Colab gave you (expect **Tesla T4**, ~15 GB memory).
If PyTorch can't see a GPU, stop here and switch the runtime type: training Pegasus on a CPU would take days.

In [ ]:
!nvidia-smi
!python -c "import torch; assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'; print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))"
assert _exit_code == 0, "No GPU available (see the message above)"

## 2. Mount Google Drive

A Colab machine is wiped when the session ends. Mounting Drive gives us somewhere permanent to save the trained model.
A pop-up asks you to approve access. It comes early on purpose, so you can approve it and then leave training running.
In the pop-up, pick the **same Google account** as Colab and tick **all** permission boxes ("Select all").

If the mount fails (`credential propagation was unsuccessful`), the notebook **carries on**: training doesn't need Drive.
Step 7 tries to mount again before saving, so be back at your computer when training ends.

In [ ]:
from google.colab import drive

try:
    drive.mount("/content/drive")
except Exception as error:  # training doesn't need Drive; step 7 mounts again before saving
    print(f"Drive mount failed: {error}")
    print("Continuing without Drive. Step 7 will ask again before saving the model.")

## 3. Clone the repo and install packages

We install everything in `requirements.txt` **except `torch`**. Colab's preinstalled PyTorch is built for its own CUDA driver.
Replacing it would download ~2–3 GB and risk a driver mismatch. The project works with any recent PyTorch 2.x.

`pip install -e .` makes `import textSummarizer` work, exactly like on your laptop.
If Colab offers to "restart session" after the install, you can ignore it: the pipeline runs in fresh processes.

In [ ]:
import os

%cd /content
if os.path.isdir("text-summarizer"):
    !git -C text-summarizer pull
else:
    !git clone {REPO_URL}
assert _exit_code == 0, "git clone/pull failed"
%cd /content/text-summarizer
!git log --oneline -1

!grep -v "^torch==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
assert _exit_code == 0, "pip install -r requirements failed"
!pip install -q -e .
assert _exit_code == 0, "pip install -e . failed"
!python -c "import torch, transformers, datasets; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__)"

## 4. CUDA out of memory? Low-memory settings

**Leave both switches off for the first run.** Use this cell only if step 6 crashes with `CUDA out of memory`. In that case:

1. Set `LOW_MEMORY = True`. This switches the optimizer to **Adafactor**, which stores far less state than AdamW,
   and turns on **gradient checkpointing**, which recomputes activations instead of storing them: ~20–30% slower, much less memory.
2. If it still runs out of memory, also set `MAX_INPUT_LENGTH = 384`. Dialogues are cut shorter, which saves activation memory.
   Step 5 re-tokenizes with the new length.
3. Click this cell, then **Runtime → Run cell and below**.

The edits only change `params.yaml` in this Colab copy, not your repo.

In [ ]:
import re
from pathlib import Path

LOW_MEMORY = False       # True: optim adafactor + gradient_checkpointing true
MAX_INPUT_LENGTH = None  # e.g. 384 if LOW_MEMORY alone isn't enough

params_file = Path("params.yaml")
params = params_file.read_text()
if LOW_MEMORY:
    params = re.sub(r"(optim:) \S+", r"\g<1> adafactor", params)
    params = re.sub(r"(gradient_checkpointing:) \S+", r"\g<1> true", params)
if MAX_INPUT_LENGTH:
    params = re.sub(r"(max_input_length:) \d+", rf"\g<1> {MAX_INPUT_LENGTH}", params)
params_file.write_text(params)

!grep -E "optim:|gradient_checkpointing:|max_input_length:" params.yaml

## 5. Ingestion and tokenization

- **Ingestion** downloads SAMSum (~10 MB) into `artifacts/data_ingestion/`.
- **Transformation** tokenizes every row with the **Pegasus** tokenizer.

`--no-smoke` overrides `smoke_test.enabled: true` in `params.yaml`, so the full data and the Pegasus checkpoint are used.

In [ ]:
!python main.py --stage ingestion --no-smoke
assert _exit_code == 0, "Ingestion failed (see the log above)"
!python main.py --stage transformation --no-smoke
assert _exit_code == 0, "Transformation failed (see the log above)"

## 6. Train

Fine-tunes `google/pegasus-cnn_dailymail` on the 14,731 training dialogues for 1 epoch:
batch size 1 × 16 gradient-accumulation steps = an effective batch of 16, so about **920 optimizer steps**.

What to watch:
- A `{'loss': ...}` line every 10 steps. It should **trend down**, roughly from ~3 toward ~1.5. It's noisy step to step; look at the trend.
- The learning rate climbs during the first 500 steps (warmup), then decays.
- At the end, the **validation loss** on 818 held-out dialogues is logged. It should be in the same range as the last training losses.

If this crashes with `CUDA out of memory`, go to step 4.

In [ ]:
!python main.py --stage training --no-smoke
assert _exit_code == 0, "Training failed. CUDA out of memory? See step 4"

## 7. Save the model

Copies `artifacts/model_trainer/` (model ~2.3 GB + tokenizer) to `MyDrive/text-summarizer/model_trainer`,
plus the run log (training and validation losses) to `MyDrive/text-summarizer/running_logs.log`.

**If Drive won't mount**, the cell packs the model into `/content/model_trainer.zip` instead. Download it from the
**Files** sidebar (folder icon, left; go up to `/content`): right-click → **Download**. Colab shows only a small
progress ring, not a percentage. Don't close this tab until the download has finished: the model only exists here.

In [ ]:
import shutil
from google.colab import drive

if not os.path.ismount("/content/drive"):
    try:
        drive.mount("/content/drive")
    except Exception as error:
        print(f"Drive mount failed: {error}")

if os.path.ismount("/content/drive"):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copytree("artifacts/model_trainer", f"{DRIVE_DIR}/model_trainer", dirs_exist_ok=True)
    shutil.copy("logs/running_logs.log", f"{DRIVE_DIR}/running_logs.log")
    !ls -lh {DRIVE_DIR}/model_trainer/model {DRIVE_DIR}/model_trainer/tokenizer
else:
    # -0 stores without compressing: model weights barely compress, so this takes seconds instead of minutes.
    !cd artifacts && zip -q -r -0 /content/model_trainer.zip model_trainer -x "model_trainer/smoke/*"
    !ls -lh /content/model_trainer.zip
    print("NOT saved to Drive. Download /content/model_trainer.zip from the Files sidebar before closing this tab.")

## 8. Sanity check: summarize 2 test dialogues

Loads the **fine-tuned** model and summarizes the first 2 test dialogues, which it never saw during training.
It reuses the evaluation code, so the settings match step 9 exactly (generation settings from `params.yaml`,
`<n>` line breaks replaced with spaces).

The summaries should be short, third-person, and name the people in the chat, like the human reference.

In [ ]:
%%writefile /content/sanity_check.py
import torch
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from textSummarizer.components.model_evaluation import generate_summaries
from textSummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager(smoke=False).get_model_evaluation_config()
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_path)
model = AutoModelForSeq2SeqLM.from_pretrained(config.model_path).to(device).eval()

test = load_from_disk(config.data_path)["test"].select(range(2))
for example, summary in zip(test, generate_summaries(model, tokenizer, list(test["dialogue"]), config)):
    print("=" * 80)
    print(example["dialogue"])
    print()
    print(f"REFERENCE: {example['summary']}")
    print(f"MODEL:     {summary}")

In [ ]:
!python /content/sanity_check.py
assert _exit_code == 0, "Sanity check failed (see the error above)"

## 9. Evaluate with ROUGE and compare with the base model

1. **Evaluation stage:** the fine-tuned model summarizes 100 test dialogues (`evaluation.num_samples`), scored with
   ROUGE-1, ROUGE-2, ROUGE-L and ROUGE-Lsum. It writes `metrics.csv`, plus `samples.csv` with 10 dialogues to read by eye.
2. **Comparison:** the original `google/pegasus-cnn_dailymail` (news-only, never saw a chat) and the fine-tuned model
   summarize the **same** 100 dialogues with the **same** settings, written to `comparison.csv`.

About 5–10 minutes on a T4. The three CSV files then download to your laptop. Put them in `artifacts/model_evaluation/`.

What to expect: the fine-tuned model should clearly beat the base model, with a fine-tuned ROUGE-1 of roughly 0.40 or more.

In [ ]:
!python main.py --stage evaluation --no-smoke
assert _exit_code == 0, "Evaluation failed (see the log above)"
!python research/compare_models.py
assert _exit_code == 0, "Comparison failed (see the log above)"

In [ ]:
from google.colab import files

for name in ["metrics.csv", "samples.csv", "comparison.csv"]:
    files.download(f"artifacts/model_evaluation/{name}")

## Done

1. Get the model to your laptop: download `text-summarizer/model_trainer` from Google Drive, or `model_trainer.zip`
   from step 7. Unzip it into `artifacts/`, so you have `artifacts/model_trainer/model/` and `artifacts/model_trainer/tokenizer/`.
2. Put the three CSV files from step 9 in `artifacts/model_evaluation/`.
3. Read `samples.csv` and judge the summaries yourself: count the ones with a wrong fact or the wrong person.